# Task 1 — Claim Detection training on Colab

In [ ]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-d985ff2e-44f7-a5f2-0d3e-9783b0b07677)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/nlp/nlp-final-project'
assert os.path.exists(f'{DRIVE_DIR}/task1_colab.zip'), f'Upload task1_colab.zip to {DRIVE_DIR} first'

Mounted at /content/drive


In [ ]:
!rm -rf /content/project && mkdir -p /content/project
!unzip -q "{DRIVE_DIR}/task1_colab.zip" -d /content/project
%cd /content/project
!ls -R data src | head -30

/content/project
data:
processed

data/processed:
task1_meta.json
task1_test.parquet
task1_train.parquet
task1_val.parquet

src:
__init__.py
task1_claim_detection

src/task1_claim_detection:
baseline.py
domain_gap.py
infer.py
__init__.py
train.py


In [ ]:
!pip install -q "transformers>=4.46" accelerate sentencepiece
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

torch 2.14.1+cu130 | transformers 5.22.1 | cuda True


## 1. Main run — factual (CFS + UFS) vs non-factual  (plan 1.2)

In [ ]:
!python -m src.task1_claim_detection.train --model roberta-base --label-col label --batch-size 32 --grad-accum 1

Run task1_roberta-base_label | class weights [0.7667224716536032, 1.4373038516405134]
config.json: 100% 481/481 [00:00<00:00, 2.07MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 124kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 5.66MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 6.81MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 8.40MB/s]

model.safetensors: downloading bytes:  62% 309M/499M [00:01<00:00, 279MB/s, 25.5MB/s  ]
model.safetensors: reconstructing file:  67% 335M/499M [00:04<00:02, 75.7MB/s]
model.safetensors: downloading bytes: 100% 335M/335M [00:04<00:00, 70.8MB/s, 30.3MB/s  ]
model.safetensors: reconstructing file: 100% 499M/499M [00:04<00:00, 105MB/s, 43.8MB/s  ]
Loading weights: 100% 197/197 [00:00<00:00, 4502.12it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNE

## 2. Comparison run — original check-worthiness mapping, CFS only  (plan 1.3)

In [ ]:
!python -m src.task1_claim_detection.train --model roberta-base --label-col label_checkworthy --batch-size 32 --grad-accum 1

Run task1_roberta-base_label_checkworthy | class weights [0.6587016213389121, 2.0752832131822863]
Loading weights: 100% 197/197 [00:00<00:00, 11791.84it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
{'train': 20151, 'val': 2240, 'test': 1032}


In [ ]:
!unzip -oq "{DRIVE_DIR}/task1_colab.zip" 'src/*' -d /content/project


## 3. (Optional) Second backbone from the plan — deberta-v3-small
Skip this cell if you are short on time or GPU quota.

In [ ]:
!python -m src.task1_claim_detection.train --model microsoft/deberta-v3-small --label-col label --batch-size 32 --grad-accum 1

Run task1_deberta-v3-small_label | class weights [0.7667224716536032, 1.4373038516405134]
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100% 102/102 [00:00<00:00, 25822.01it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predict

## 4. Summary of all runs

In [ ]:
import glob, json
rows = []
for path in sorted(glob.glob('models/*/metrics.json')):
    m = json.load(open(path))
    if 'test@tuned' not in m:      # skip the tf-idf baseline file if present
        continue
    t = m['test@tuned']
    rows.append((m['run'], m['tuned_threshold'], t['macro_f1'], t['precision'], t['recall'], t['roc_auc']))
print(f"{'run':<45}{'thr':>6}{'macroF1':>9}{'prec':>8}{'recall':>8}{'rocAUC':>8}")
for r in rows:
    print(f"{r[0]:<45}{r[1]:>6.2f}{r[2]:>9.4f}{r[3]:>8.4f}{r[4]:>8.4f}{r[5]:>8.4f}")

run                                             thr  macroF1    prec  recall  rocAUC
task1_deberta-v3-small_label                   0.58   0.9789  0.9701  0.9701  0.9971
task1_roberta-base_label                       0.60   0.9755  0.9575  0.9734  0.9975
task1_roberta-base_label_checkworthy           0.53   0.9689  0.9388  0.9664  0.9969


## 5. Save trained models to Drive

In [ ]:
!rm -f "{DRIVE_DIR}/task1_models.zip"
!zip -qr "{DRIVE_DIR}/task1_models.zip" models -x "models/*/trainer/*"
!ls -lh "{DRIVE_DIR}/task1_models.zip"

-rw------- 1 root root 1.3G Nov 18 04:35 /content/drive/MyDrive/nlp/nlp-final-project/task1_models.zip
